
# Final Sweet-Spot Hardware Evaluation

**Purpose:** Rebuild the final analysis from the actual IBM Quantum jobs and the saved experiment artifacts, with **one and only one Kingston job** used in the primary analysis.

### Primary hardware jobs
- `ibm_fez`: `dafgfb5nj4cs73agbfo0`
- `ibm_marrakesh`: `dafpe83dd5gc73da62o0`
- `ibm_kingston`: **fresh Kingston job** `dafpntd1ierc738nnmpg`

### Kingston duplicate handling
Two Kingston jobs exist:
- Old: `dafgfgl1ierc738nbdd0`
- Fresh: `dafpntd1ierc738nnmpg`

**Only the fresh Kingston job is used in the final primary experiment. The old Kingston job is recorded as excluded and is not mixed into the main statistics.**

### Experiment
Temporal **30-day Sweet-Spot** vs **10 topology-valid random controls**, 11 circuits/backend, 256 shots/circuit.

### Outputs
Everything is written into:

```text
<ibm_circuits_project>/final_results/
```

The notebook creates raw collected data, normalized data, CSV tables, statistical evaluations, figures, and a final research conclusion.


In [1]:

# 1. Imports and deterministic configuration

from pathlib import Path
import os, sys, json, ast, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore")

RANDOM_SEED = 20260906
rng = np.random.default_rng(RANDOM_SEED)

PRIMARY_JOBS = {
    "ibm_fez": "dafgfb5nj4cs73agbfo0",
    "ibm_marrakesh": "dafpe83dd5gc73da62o0",
    "ibm_kingston": "dafpntd1ierc738nnmpg",
}

OLD_KINGSTON_JOB = "dafgfgl1ierc738nbdd0"

EXPECTED_BACKENDS = ["ibm_fez", "ibm_marrakesh", "ibm_kingston"]
EXPECTED_CIRCUITS = 11
EXPECTED_CONTROLS = 10
EXPECTED_SHOTS = 256

MODEL_NAME = "temporal_30day"
PREDICTION_DATE = "2026-09-06"

print("Configuration loaded.")
print("Primary jobs:", PRIMARY_JOBS)
print("Excluded old Kingston job:", OLD_KINGSTON_JOB)


Configuration loaded.
Primary jobs: {'ibm_fez': 'dafgfb5nj4cs73agbfo0', 'ibm_marrakesh': 'dafpe83dd5gc73da62o0', 'ibm_kingston': 'dafpntd1ierc738nnmpg'}
Excluded old Kingston job: dafgfgl1ierc738nbdd0


In [2]:

# 2. Locate the actual ibm_circuits project and create FINAL output folder

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("/mnt/data"),
    ]

    for c in candidates:
        c = c.resolve()
        if (c / "results").exists() or (c / "run_level1.py").exists() or (c / "select_level2_candidates.py").exists():
            return c

    # If running from JupyterLab, walk upward.
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "results").exists() or (parent / "run_level1.py").exists():
            return parent

    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
FINAL_DIR = PROJECT_ROOT / "final_results"
RAW_DIR = FINAL_DIR / "raw_data"
TABLE_DIR = FINAL_DIR / "tables"
FIG_DIR = FINAL_DIR / "figures"

for d in [FINAL_DIR, RAW_DIR, TABLE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT :", PROJECT_ROOT)
print("FINAL RESULTS:", FINAL_DIR)
print("RAW DATA     :", RAW_DIR)
print("TABLES       :", TABLE_DIR)
print("FIGURES      :", FIG_DIR)


PROJECT ROOT : D:\Downloads\uday projects\ibm_circuits
FINAL RESULTS: D:\Downloads\uday projects\ibm_circuits\final_results
RAW DATA     : D:\Downloads\uday projects\ibm_circuits\final_results\raw_data
TABLES       : D:\Downloads\uday projects\ibm_circuits\final_results\tables
FIGURES      : D:\Downloads\uday projects\ibm_circuits\final_results\figures


In [3]:

# 3. Find existing local experiment artifacts

def locate(name):
    candidates = [
        PROJECT_ROOT / "results" / name,
        PROJECT_ROOT / "results" / "sweet_spot" / name,
        PROJECT_ROOT / name,
        Path.cwd() / name,
        Path("/mnt/data") / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

artifact_names = [
    "sweet_spot_level2_candidates.csv",
    "sweet_spot_level2_hardware_results.json",
    "sweet_spot_level2_ibm_fez_result.json",
    "sweet_spot_level2_ibm_marrakesh_result.json",
    "sep6_all_temporal_model_predictions.csv",
    "temporal_model_comparison.csv",
    "seven_vs_thirty_day.csv",
]

artifact_locations = {n: locate(n) for n in artifact_names}

for n, p in artifact_locations.items():
    print(f"{n:55s} -> {p}")


sweet_spot_level2_candidates.csv                        -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_candidates.csv
sweet_spot_level2_hardware_results.json                 -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_hardware_results.json
sweet_spot_level2_ibm_fez_result.json                   -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_ibm_fez_result.json
sweet_spot_level2_ibm_marrakesh_result.json             -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot_level2_ibm_marrakesh_result.json
sep6_all_temporal_model_predictions.csv                 -> D:\Downloads\uday projects\ibm_circuits\results\sweet_spot\sep6_all_temporal_model_predictions.csv
temporal_model_comparison.csv                           -> None
seven_vs_thirty_day.csv                                 -> None



## 4. Collect the actual Kingston primary data

This section **does not submit a job**.

It connects to IBM Quantum and retrieves the completed fresh Kingston job. If a local result file already exists and matches the fresh job ID, it is used as a cross-check; otherwise the completed IBM job is retrieved.

The old Kingston job is deliberately excluded from the primary experiment.


In [4]:

# 4A. IBM Runtime retrieval helpers

def get_ibm_service():
    from qiskit_ibm_runtime import QiskitRuntimeService

    # Existing saved IBM credentials are preferred.
    try:
        return QiskitRuntimeService()
    except Exception as e:
        print("QiskitRuntimeService() failed:", repr(e))
        print("If needed, authenticate in the normal IBM Quantum way and rerun this cell.")
        raise

def extract_counts_from_pub(pub_result):
    # SamplerV2 normally exposes measurement data through data.meas.
    data = getattr(pub_result, "data", None)
    if data is None:
        raise ValueError("PUB result has no .data attribute.")

    meas = getattr(data, "meas", None)
    if meas is None:
        # Fallback: locate a measurement-like field.
        for name in dir(data):
            if name.startswith("_"):
                continue
            try:
                obj = getattr(data, name)
                if hasattr(obj, "get_counts"):
                    meas = obj
                    break
            except Exception:
                pass

    if meas is None or not hasattr(meas, "get_counts"):
        raise ValueError("Could not locate a measurement result with get_counts().")

    counts = meas.get_counts()
    return {str(k): int(v) for k, v in counts.items()}

def ghz_fidelity(counts):
    total = sum(counts.values())
    if total <= 0:
        return np.nan
    return (counts.get("000", 0) + counts.get("111", 0)) / total

print("IBM retrieval helpers ready.")


IBM retrieval helpers ready.


In [5]:

# 4B. Retrieve the fresh Kingston job only

service = get_ibm_service()

fresh_kingston_id = PRIMARY_JOBS["ibm_kingston"]

if fresh_kingston_id == OLD_KINGSTON_JOB:
    raise RuntimeError("Safety check failed: primary Kingston job equals the excluded old job.")

kingston_job = service.job(fresh_kingston_id)

print("Kingston job:", fresh_kingston_id)
print("Status:", kingston_job.status())

if str(kingston_job.status()).upper() not in {"DONE", "COMPLETED"}:
    print("WARNING: Kingston primary job is not currently DONE.")
    print("Do not submit another job from this notebook.")
else:
    print("Fresh Kingston job is complete. Retrieving result...")

    kingston_result = kingston_job.result()
    kingston_pubs = list(kingston_result)

    if len(kingston_pubs) != EXPECTED_CIRCUITS:
        raise RuntimeError(
            f"Expected {EXPECTED_CIRCUITS} Kingston PUB results, got {len(kingston_pubs)}."
        )

    kingston_counts = [extract_counts_from_pub(pub) for pub in kingston_pubs]

    kingston_raw = {
        "backend": "ibm_kingston",
        "job_id": fresh_kingston_id,
        "status": str(kingston_job.status()),
        "circuits": len(kingston_counts),
        "shots_expected": EXPECTED_SHOTS,
        "counts": kingston_counts,
    }

    with open(RAW_DIR / "ibm_kingston_fresh_job_raw.json", "w", encoding="utf-8") as f:
        json.dump(kingston_raw, f, indent=2)

    print("Saved:", RAW_DIR / "ibm_kingston_fresh_job_raw.json")


QiskitRuntimeService() failed: AccountNotFoundError("Unable to find account. Please make sure an account with the channel name 'ibm_quantum_platform' is saved.")
If needed, authenticate in the normal IBM Quantum way and rerun this cell.


AccountNotFoundError: "Unable to find account. Please make sure an account with the channel name 'ibm_quantum_platform' is saved."


## 5. Build one clean primary dataset

FEZ and Marrakesh are loaded from the already completed primary result artifacts. Kingston is loaded from the **fresh** IBM job above.

No old Kingston observation enters this dataset.


In [ ]:

# 5A. Load primary local hardware data

def load_primary_local():
    p = locate("sweet_spot_level2_hardware_results.json")
    if p is not None:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        return obj

    p = locate("sweet_spot_level2_ibm_fez_result.json")
    q = locate("sweet_spot_level2_ibm_marrakesh_result.json")
    if p is None or q is None:
        raise FileNotFoundError("Could not locate FEZ/Marrakesh primary result artifacts.")

    with open(p, "r", encoding="utf-8") as f:
        fez = json.load(f)
    with open(q, "r", encoding="utf-8") as f:
        mar = json.load(f)

    return {"backends": [fez, mar]}

local_obj = load_primary_local()
print("Loaded local primary hardware artifact.")


In [ ]:

# 5B. Normalize FEZ/Marrakesh records from the saved hardware results

def flatten_backend_object(obj):
    rows = []

    # Common structure: {"backends": [...]} or a single backend dict.
    backends = obj.get("backends") if isinstance(obj, dict) else None
    if backends is None:
        if isinstance(obj, dict) and "backend" in obj:
            backends = [obj]
        else:
            backends = []

    for b in backends:
        backend = b.get("backend")
        records = (
            b.get("circuits")
            or b.get("results")
            or b.get("circuit_results")
            or []
        )
        for r in records:
            rr = dict(r)
            rr["backend"] = backend
            rows.append(rr)

    return rows

local_rows = flatten_backend_object(local_obj)

# If the JSON structure differs, use the CSV primary artifact as a reliable fallback.
if not local_rows:
    p = locate("sweet_spot_level2_hardware_results.json")
    if p is None:
        raise RuntimeError("Could not identify FEZ/Marrakesh circuit records.")
    raise RuntimeError(
        "The local hardware-results JSON structure was not recognized. "
        "Inspect the JSON structure before proceeding."
    )

local_df = pd.DataFrame(local_rows)

print(local_df.shape)
print(local_df.columns.tolist())


In [ ]:

# 5C. Normalize column names and add Kingston fresh records

def normalize_columns(df):
    df = df.copy()

    aliases = {
        "qubits": "physical_qubits",
        "fidelity": "ghz_fidelity",
        "depth": "transpiled_depth",
        "viability": "mean_viability",
    }
    for a, b in aliases.items():
        if a in df.columns and b not in df.columns:
            df[b] = df[a]

    required = [
        "backend", "circuit_index", "group", "trial", "physical_qubits",
        "mean_viability", "shots", "job_id", "ghz_fidelity",
        "transpiled_depth", "is_sweet_spot"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Primary hardware data missing columns: {missing}")

    return df

local_df = normalize_columns(local_df)

# Keep only the two known completed primary backends.
local_df = local_df[local_df["backend"].isin(["ibm_fez", "ibm_marrakesh"])].copy()

if "kingston_counts" in globals():
    candidates_path = locate("sweet_spot_level2_candidates.csv")
    if candidates_path is None:
        raise FileNotFoundError("Candidate manifest is required to label Kingston circuits.")

    candidates = pd.read_csv(candidates_path)
    k = candidates[candidates["backend"].eq("ibm_kingston")].copy()

    if len(k) != EXPECTED_CIRCUITS:
        raise RuntimeError(f"Expected 11 Kingston candidate rows, got {len(k)}.")

    # Candidate manifest should already be ordered by circuit index.
    k = k.sort_values("circuit_index").reset_index(drop=True)

    if len(kingston_counts) != len(k):
        raise RuntimeError("Kingston PUB count and candidate manifest count differ.")

    k_rows = []
    for i, counts in enumerate(kingston_counts):
        c = k.iloc[i]
        k_rows.append({
            "backend": "ibm_kingston",
            "circuit_index": int(c["circuit_index"]),
            "group": c["group"],
            "trial": int(c["trial"]),
            "model": c.get("model", "random" if c["group"] == "random_control" else MODEL_NAME),
            "physical_qubits": c["physical_qubits"],
            "mean_viability": c.get("mean_viability", np.nan),
            "shots": int(sum(counts.values())),
            "job_id": fresh_kingston_id,
            "counts": str(counts),
            "ghz_fidelity": ghz_fidelity(counts),
            "transpiled_depth": np.nan,
            "is_sweet_spot": c["group"] == "sweet_spot",
        })

    kingston_df = pd.DataFrame(k_rows)
else:
    raise RuntimeError(
        "Fresh Kingston data were not collected. "
        "Run the Kingston retrieval cell successfully before continuing."
    )

primary = pd.concat([local_df, kingston_df], ignore_index=True)

# Hard safety checks.
if OLD_KINGSTON_JOB in set(primary["job_id"].astype(str)):
    raise RuntimeError("OLD KINGSTON JOB LEAKED INTO PRIMARY DATASET.")

primary.to_csv(TABLE_DIR / "primary_hardware_all_backends.csv", index=False)

print("PRIMARY DATASET")
print(primary.groupby("backend").size())
print("Saved:", TABLE_DIR / "primary_hardware_all_backends.csv")


In [ ]:

# 6. Structural validation before statistics

errors = []

for backend in EXPECTED_BACKENDS:
    d = primary[primary["backend"].eq(backend)].copy()

    if len(d) != EXPECTED_CIRCUITS:
        errors.append(f"{backend}: expected 11 circuits, got {len(d)}")

    if d["circuit_index"].nunique() != EXPECTED_CIRCUITS:
        errors.append(f"{backend}: circuit_index is not unique")

    if d["shots"].nunique() != 1 or int(d["shots"].iloc[0]) != EXPECTED_SHOTS:
        errors.append(f"{backend}: shots are not exactly 256")

    if d["group"].eq("sweet_spot").sum() != 1:
        errors.append(f"{backend}: expected exactly one Sweet-Spot circuit")

    if d["group"].eq("random_control").sum() != EXPECTED_CONTROLS:
        errors.append(f"{backend}: expected exactly ten random controls")

if errors:
    print("\n".join(errors))
    raise RuntimeError("Primary dataset validation failed.")

print("PRIMARY DATASET VALIDATION: PASS")



## 7. Define what the Sweet-Spot actually means

A **Sweet-Spot** is not simply "a high-scoring individual qubit."

The experiment selects a **topology-valid physical 3-qubit chain** whose three qubits have the highest model-predicted viability collectively, using the **Temporal 30-day model**. The chain score is the mean predicted viability of its three physical qubits.

The random controls are also topology-valid 3-qubit chains, so the hardware comparison asks:

> Does deliberately selecting a chain using the model produce better GHZ hardware fidelity than selecting topology-valid chains at random?

The model score is a **selection/ranking signal**, not a claim that predicted viability numerically equals GHZ fidelity.


In [ ]:

# 8. Load temporal model-selection evidence

tm_path = locate("temporal_model_comparison.csv")
sv_path = locate("seven_vs_thirty_day.csv")

if tm_path is None:
    raise FileNotFoundError("temporal_model_comparison.csv not found.")

temporal = pd.read_csv(tm_path)

if sv_path is not None:
    seven_vs_thirty = pd.read_csv(sv_path)
else:
    seven_vs_thirty = pd.DataFrame()

display(temporal)
if not seven_vs_thirty.empty:
    display(seven_vs_thirty)


In [ ]:

# 9. Backend-level primary evaluation

summary_rows = []

for backend in EXPECTED_BACKENDS:
    d = primary[primary["backend"].eq(backend)].copy()
    sweet = d[d["group"].eq("sweet_spot")].iloc[0]
    controls = d[d["group"].eq("random_control")].copy()

    sf = float(sweet["ghz_fidelity"])
    cf = controls["ghz_fidelity"].astype(float).to_numpy()

    advantage = sf - cf.mean()
    pct = 100 * advantage / cf.mean()

    beaten = int(np.sum(sf > cf))
    ties = int(np.sum(np.isclose(sf, cf)))

    ranks = d["ghz_fidelity"].rank(method="min", ascending=False)
    sweet_rank = int(ranks.loc[sweet.name])

    summary_rows.append({
        "backend": backend,
        "sweet_fidelity": sf,
        "random_mean": float(cf.mean()),
        "random_median": float(np.median(cf)),
        "random_std": float(np.std(cf, ddof=1)),
        "absolute_advantage": float(advantage),
        "percent_advantage": float(pct),
        "controls_beaten": beaten,
        "controls_total": len(cf),
        "fraction_controls_beaten": beaten / len(cf),
        "ties": ties,
        "sweet_rank_of_11": sweet_rank,
        "sweet_depth": sweet["transpiled_depth"],
        "random_median_depth": float(controls["transpiled_depth"].median()) if controls["transpiled_depth"].notna().any() else np.nan,
        "predicted_mean_viability": sweet["mean_viability"],
    })

backend_summary = pd.DataFrame(summary_rows)
backend_summary.to_csv(TABLE_DIR / "backend_summary.csv", index=False)

display(backend_summary.round(6))


In [ ]:

# 10. Every-circuit ranking

ranking = primary.copy()

ranking["rank"] = (
    ranking.groupby("backend")["ghz_fidelity"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking["control_percentile"] = np.nan

for backend in EXPECTED_BACKENDS:
    d = ranking[ranking["backend"].eq(backend)]
    controls = d[d["group"].eq("random_control")]["ghz_fidelity"].to_numpy()

    mask = ranking["backend"].eq(backend) & ranking["group"].eq("sweet_spot")
    sf = float(ranking.loc[mask, "ghz_fidelity"].iloc[0])

    if len(controls):
        ranking.loc[mask, "control_percentile"] = 100 * np.mean(controls < sf)

ranking.to_csv(TABLE_DIR / "circuit_ranking.csv", index=False)

display(
    ranking[
        ["backend","circuit_index","group","physical_qubits",
         "ghz_fidelity","transpiled_depth","rank","control_percentile"]
    ].sort_values(["backend","rank"]).round(6)
)


In [ ]:

# 11. Shot uncertainty: exact binomial Wilson intervals

def wilson_interval(k, n, z=1.959963984540054):
    if n == 0:
        return np.nan, np.nan
    phat = k / n
    denom = 1 + z*z/n
    centre = (phat + z*z/(2*n)) / denom
    half = z * math.sqrt((phat*(1-phat)/n) + z*z/(4*n*n)) / denom
    return centre-half, centre+half

unc_rows = []

for _, row in primary.iterrows():
    counts = ast.literal_eval(row["counts"]) if isinstance(row["counts"], str) else row["counts"]
    n = int(sum(counts.values()))
    k = int(counts.get("000", 0) + counts.get("111", 0))
    lo, hi = wilson_interval(k, n)

    unc_rows.append({
        "backend": row["backend"],
        "circuit_index": row["circuit_index"],
        "group": row["group"],
        "physical_qubits": row["physical_qubits"],
        "success_shots_000_or_111": k,
        "shots": n,
        "ghz_fidelity": row["ghz_fidelity"],
        "wilson_95_low": lo,
        "wilson_95_high": hi,
    })

shot_uncertainty = pd.DataFrame(unc_rows)
shot_uncertainty.to_csv(TABLE_DIR / "shot_uncertainty.csv", index=False)

display(shot_uncertainty.round(6))


In [ ]:

# 12. Same-depth sensitivity analysis

depth_rows = []

for backend in EXPECTED_BACKENDS:
    d = primary[primary["backend"].eq(backend)].copy()
    sweet = d[d["group"].eq("sweet_spot")].iloc[0]
    sf = float(sweet["ghz_fidelity"])

    same = d[
        (d["group"].eq("random_control")) &
        (d["transpiled_depth"].eq(sweet["transpiled_depth"]))
    ]["ghz_fidelity"].astype(float)

    different = d[
        (d["group"].eq("random_control")) &
        (d["transpiled_depth"].ne(sweet["transpiled_depth"]))
    ]["ghz_fidelity"].astype(float)

    depth_rows.append({
        "backend": backend,
        "sweet_depth": sweet["transpiled_depth"],
        "controls_same_depth": len(same),
        "controls_different_depth": len(different),
        "sweet_fidelity": sf,
        "same_depth_control_mean": same.mean() if len(same) else np.nan,
        "same_depth_advantage": sf - same.mean() if len(same) else np.nan,
        "different_depth_control_mean": different.mean() if len(different) else np.nan,
        "all_control_mean": d[d["group"].eq("random_control")]["ghz_fidelity"].mean(),
    })

depth_sensitivity = pd.DataFrame(depth_rows)
depth_sensitivity.to_csv(TABLE_DIR / "depth_sensitivity.csv", index=False)

display(depth_sensitivity.round(6))


In [ ]:

# 13. Empirical control test and within-backend paired advantage

emp_rows = []

for backend in EXPECTED_BACKENDS:
    d = primary[primary["backend"].eq(backend)]
    sweet = float(d[d["group"].eq("sweet_spot")]["ghz_fidelity"].iloc[0])
    controls = d[d["group"].eq("random_control")]["ghz_fidelity"].astype(float).to_numpy()

    below = int(np.sum(controls < sweet))

    # Exact one-sided rank probability:
    # under exchangeability among the 11 measured circuits,
    # the Sweet-Spot rank is uniform from 1..11.
    rank = int((np.sum(controls > sweet) + 1))
    exact_p = rank / (len(controls) + 1)

    emp_rows.append({
        "backend": backend,
        "sweet_fidelity": sweet,
        "controls": len(controls),
        "controls_below_sweet": below,
        "fraction_below": below / len(controls),
        "sweet_rank": rank,
        "exact_one_sided_rank_p": exact_p,
    })

empirical = pd.DataFrame(emp_rows)
empirical.to_csv(TABLE_DIR / "empirical_control_test.csv", index=False)

advantages = backend_summary["absolute_advantage"].to_numpy(float)

# Across-backend sign test: probability of all 3 advantages being positive.
n_backend = len(advantages)
positive = int(np.sum(advantages > 0))
sign_p = sum(math.comb(n_backend, k) for k in range(positive, n_backend+1)) / (2**n_backend)

if n_backend >= 3:
    try:
        wilcoxon_stat, wilcoxon_p = stats.wilcoxon(
            advantages, alternative="greater", zero_method="wilcox"
        )
    except Exception:
        wilcoxon_stat, wilcoxon_p = np.nan, np.nan
else:
    wilcoxon_stat, wilcoxon_p = np.nan, np.nan

print("Backend advantages:", advantages)
print("All-positive sign-test p:", sign_p)
print("Wilcoxon signed-rank statistic:", wilcoxon_stat)
print("Wilcoxon one-sided p:", wilcoxon_p)

display(empirical.round(6))


In [ ]:

# 14. Bootstrap the mean backend advantage (descriptive, n=3)

B = 20000
boot_means = np.empty(B)

for i in range(B):
    sample = rng.choice(advantages, size=len(advantages), replace=True)
    boot_means[i] = sample.mean()

boot_low, boot_high = np.quantile(boot_means, [0.025, 0.975])

bootstrap_summary = pd.DataFrame([{
    "observed_mean_backend_advantage": advantages.mean(),
    "bootstrap_95_low": boot_low,
    "bootstrap_95_high": boot_high,
    "bootstrap_iterations": B,
}])

bootstrap_summary.to_csv(TABLE_DIR / "bootstrap_advantage.csv", index=False)

display(bootstrap_summary.round(6))


In [ ]:

# 15. Predicted viability vs measured Sweet-Spot fidelity

pv = backend_summary[
    ["backend","predicted_mean_viability","sweet_fidelity"]
].copy()

pv.to_csv(TABLE_DIR / "predicted_vs_hardware.csv", index=False)

if len(pv) >= 3:
    pearson = stats.pearsonr(pv["predicted_mean_viability"], pv["sweet_fidelity"])
    spearman = stats.spearmanr(pv["predicted_mean_viability"], pv["sweet_fidelity"])
else:
    pearson = (np.nan, np.nan)
    spearman = (np.nan, np.nan)

print("Pearson r:", pearson[0], "p:", pearson[1])
print("Spearman rho:", spearman[0], "p:", spearman[1])

display(pv.round(6))


In [ ]:

# 16. Direct 7-day vs 30-day model-selection evaluation

required_metrics = {
    "LOBO mean AUC",
    "LOBO minimum AUC",
    "Top-10 Jaccard",
}

m30 = temporal[temporal["window_days"].eq(30)].iloc[0]
m7 = temporal[temporal["window_days"].eq(7)].iloc[0]

model_selection = pd.DataFrame([
    {
        "metric": "LOBO mean AUC",
        "7_day": m7["lobo_mean_auc"],
        "30_day": m30["lobo_mean_auc"],
        "30_minus_7": m30["lobo_mean_auc"] - m7["lobo_mean_auc"],
    },
    {
        "metric": "LOBO minimum AUC",
        "7_day": m7["lobo_min_auc"],
        "30_day": m30["lobo_min_auc"],
        "30_minus_7": m30["lobo_min_auc"] - m7["lobo_min_auc"],
    },
    {
        "metric": "Top-10 Jaccard",
        "7_day": m7["avg_top10_jaccard"],
        "30_day": m30["avg_top10_jaccard"],
        "30_minus_7": m30["avg_top10_jaccard"] - m7["avg_top10_jaccard"],
    },
])

model_selection.to_csv(TABLE_DIR / "seven_vs_thirty_day.csv", index=False)

display(model_selection.round(6))



## 17. Research interpretation rules

The final conclusion deliberately distinguishes:

- **Predictive model quality:** held-out LOBO AUC.
- **Selection stability:** temporal Top-10 Jaccard.
- **Hardware outcome:** measured GHZ fidelity.
- **Hardware advantage:** Sweet-Spot fidelity minus the random-control mean on the same backend.
- **Statistical evidence:** descriptive and exact tests, with the limitation that there are only three independent backend-level experiments.

The analysis will **not** claim that the model has "proved universal superiority." It will state exactly what the observed experiment supports.


In [ ]:

# 18. Figure 1 — Sweet-Spot vs random-control mean

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(backend_summary))
w = 0.36

ax.bar(x - w/2, backend_summary["sweet_fidelity"], width=w, label="Sweet-Spot")
ax.bar(x + w/2, backend_summary["random_mean"], width=w, label="Random mean")

ax.set_xticks(x)
ax.set_xticklabels([b.replace("ibm_", "").upper() for b in backend_summary["backend"]])
ax.set_ylabel("GHZ fidelity")
ax.set_title("Sweet-Spot vs random-control mean")
ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "01_sweet_spot_vs_random_mean.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 19. Figure 2 — All measured circuits

fig, ax = plt.subplots(figsize=(14, 7))

offset = 0
for backend in EXPECTED_BACKENDS:
    d = ranking[ranking["backend"].eq(backend)].sort_values("circuit_index")
    xpos = np.arange(len(d)) + offset

    ax.bar(
        xpos,
        d["ghz_fidelity"],
        label=backend.replace("ibm_", "").upper()
    )

    offset += len(d) + 1

ax.set_ylabel("GHZ fidelity")
ax.set_title("All measured circuits per backend")
ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "02_all_circuit_fidelities.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 20. Figure 3 — Backend advantage

fig, ax = plt.subplots(figsize=(10, 6))

labels = [b.replace("ibm_", "").upper() for b in backend_summary["backend"]]
ax.bar(labels, backend_summary["absolute_advantage"])

ax.axhline(0, linewidth=1)
ax.set_ylabel("Sweet-Spot fidelity − random mean")
ax.set_title("Sweet-Spot hardware advantage")

fig.tight_layout()
fig.savefig(FIG_DIR / "03_backend_advantage.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 21. Figure 4 — Temporal model predictive performance

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    temporal["window_days"],
    temporal["lobo_mean_auc"],
    marker="o",
    label="Mean LOBO AUC"
)
ax.plot(
    temporal["window_days"],
    temporal["lobo_min_auc"],
    marker="s",
    label="Minimum LOBO AUC"
)

ax.set_xlabel("Historical training window (days)")
ax.set_ylabel("AUC")
ax.set_title("Temporal model predictive performance")
ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "04_temporal_auc.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 22. Figure 5 — Temporal ranking stability

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    temporal["window_days"],
    temporal["avg_top10_jaccard"],
    marker="o"
)

ax.set_xlabel("Historical training window (days)")
ax.set_ylabel("Average Top-10 Jaccard")
ax.set_title("Temporal ranking stability")

fig.tight_layout()
fig.savefig(FIG_DIR / "05_temporal_ranking_stability.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 23. Figure 6 — 7-day vs 30-day

metrics = ["lobo_mean_auc", "lobo_min_auc", "avg_top10_jaccard"]
labels = ["Mean LOBO AUC", "Minimum LOBO AUC", "Top-10 Jaccard"]

v7 = [m7[m] for m in metrics]
v30 = [m30[m] for m in metrics]

fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(labels))
w = 0.36

ax.bar(x - w/2, v7, width=w, label="7-day")
ax.bar(x + w/2, v30, width=w, label="30-day")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Metric value")
ax.set_title("Why 30-day was selected over 7-day")
ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "06_7day_vs_30day.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 24. Figure 7 — Predicted viability vs measured hardware fidelity

fig, ax = plt.subplots(figsize=(10, 7))

for _, row in pv.iterrows():
    ax.scatter(
        row["predicted_mean_viability"],
        row["sweet_fidelity"],
        s=180
    )
    ax.annotate(
        row["backend"].replace("ibm_", "").upper(),
        (row["predicted_mean_viability"], row["sweet_fidelity"]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=12
    )

ax.set_xlabel("Predicted Sweet-Spot mean viability")
ax.set_ylabel("Measured GHZ fidelity")
ax.set_title("Predicted viability vs measured hardware fidelity")

fig.tight_layout()
fig.savefig(FIG_DIR / "07_viability_vs_hardware_fidelity.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 25. Figure 8 — Same-depth sensitivity

sd = depth_sensitivity.copy()

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(sd))
w = 0.36

ax.bar(
    x - w/2,
    sd["sweet_fidelity"],
    width=w,
    label="Sweet-Spot"
)
ax.bar(
    x + w/2,
    sd["same_depth_control_mean"],
    width=w,
    label="Same-depth random mean"
)

ax.set_xticks(x)
ax.set_xticklabels([b.replace("ibm_", "").upper() for b in sd["backend"]])
ax.set_ylabel("GHZ fidelity")
ax.set_title("Same-transpilation-depth sensitivity")
ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "08_same_depth_sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 26. Figure 9 — Kingston fresh primary result

k = primary[primary["backend"].eq("ibm_kingston")].sort_values("circuit_index")

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    np.arange(len(k)),
    k["ghz_fidelity"]
)

sweet_mask = k["group"].eq("sweet_spot").to_numpy()
for bar, is_sweet in zip(bars, sweet_mask):
    if is_sweet:
        bar.set_linewidth(3)

ax.set_xticks(np.arange(len(k)))
ax.set_xticklabels(k["circuit_index"])
ax.set_xlabel("Circuit index")
ax.set_ylabel("GHZ fidelity")
ax.set_title("Fresh Kingston: Sweet-Spot and random controls")

fig.tight_layout()
fig.savefig(FIG_DIR / "09_kingston_primary.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# 27. Final pooled descriptive analysis

all_controls = primary[primary["group"].eq("random_control")]["ghz_fidelity"].astype(float)
all_sweet = primary[primary["group"].eq("sweet_spot")]["ghz_fidelity"].astype(float)

pooled = pd.DataFrame([{
    "backends": len(EXPECTED_BACKENDS),
    "sweet_spot_observations": len(all_sweet),
    "random_control_observations": len(all_controls),
    "sweet_mean_across_backends": all_sweet.mean(),
    "random_mean_pooled": all_controls.mean(),
    "pooled_absolute_difference": all_sweet.mean() - all_controls.mean(),
}])

pooled.to_csv(TABLE_DIR / "pooled_descriptive_summary.csv", index=False)

display(pooled.round(6))


In [ ]:

# 28. Final automated research conclusion

mean_adv = float(backend_summary["absolute_advantage"].mean())
mean_pct = float(backend_summary["percent_advantage"].mean())
positive_backends = int(np.sum(backend_summary["absolute_advantage"] > 0))
n_backends = len(backend_summary)

rank_list = backend_summary["sweet_rank_of_11"].astype(int).tolist()

same_depth_positive = int(np.sum(
    depth_sensitivity["same_depth_advantage"].dropna().to_numpy(float) > 0
))
same_depth_total = int(depth_sensitivity["same_depth_advantage"].notna().sum())

winner30_auc = float(m30["lobo_mean_auc"] > m7["lobo_mean_auc"])
winner30_stability = float(m30["avg_top10_jaccard"] > m7["avg_top10_jaccard"])

conclusion = f'''
FINAL HARDWARE + MODEL EVALUATION
=================================

PRIMARY EXPERIMENT
Temporal 30-day Sweet-Spot vs topology-valid random controls.
Backends: {", ".join(EXPECTED_BACKENDS)}
Circuits/backend: {EXPECTED_CIRCUITS}
Shots/circuit: {EXPECTED_SHOTS}

KINGSTON JOB POLICY
-------------------
Fresh Kingston job used: {PRIMARY_JOBS["ibm_kingston"]}
Old Kingston job excluded: {OLD_KINGSTON_JOB}
The old Kingston job was NOT mixed into the primary experiment.

HARDWARE RESULT
---------------
Sweet-Spot beat the random-control mean on {positive_backends}/{n_backends} backends.
Mean backend-level absolute advantage: {mean_adv:.6f}
Mean backend-level percentage advantage: {mean_pct:.2f}%
Sweet-Spot ranks among the 11 circuits: {rank_list}

SAME-DEPTH SENSITIVITY
----------------------
Positive Sweet-Spot advantage on {same_depth_positive}/{same_depth_total}
backend(s) where same-depth controls were available.

MODEL SELECTION
---------------
30-day mean LOBO AUC: {m30["lobo_mean_auc"]:.6f}
7-day mean LOBO AUC:  {m7["lobo_mean_auc"]:.6f}
30-day minus 7-day AUC: {m30["lobo_mean_auc"] - m7["lobo_mean_auc"]:.6f}

30-day minimum LOBO AUC: {m30["lobo_min_auc"]:.6f}
7-day minimum LOBO AUC:  {m7["lobo_min_auc"]:.6f}

30-day Top-10 Jaccard: {m30["avg_top10_jaccard"]:.6f}
7-day Top-10 Jaccard:  {m7["avg_top10_jaccard"]:.6f}

INTERPRETATION
--------------
The experiment tests whether a telemetry-derived ranking can select
physical 3-qubit regions that perform better on a GHZ benchmark than
topology-valid random controls.

A positive result across all three primary backends is consistent with
the hypothesis that the model-selected Sweet-Spot contains hardware
regions with better observed GHZ fidelity than randomly selected valid
controls.

However, this is evidence rather than universal proof. The independent
backend count is small (n={n_backends}), each backend has only one
Sweet-Spot circuit and ten controls, and hardware execution introduces
shot noise and possible routing/depth effects.

The 30-day model is preferred over the 7-day model because it has both
higher held-out LOBO AUC and substantially higher temporal Top-10
ranking stability in the pre-hardware model-selection experiment.

The model's predicted viability should be interpreted as a ranking
signal, not as a numerically calibrated prediction of GHZ fidelity.
'''

print(conclusion)

with open(FINAL_DIR / "FINAL_RESEARCH_CONCLUSION.txt", "w", encoding="utf-8") as f:
    f.write(conclusion)

final_summary = {
    "primary_jobs": PRIMARY_JOBS,
    "excluded_old_kingston_job": OLD_KINGSTON_JOB,
    "backends": EXPECTED_BACKENDS,
    "circuits_per_backend": EXPECTED_CIRCUITS,
    "shots_per_circuit": EXPECTED_SHOTS,
    "positive_backend_count": positive_backends,
    "backend_count": n_backends,
    "mean_backend_advantage": mean_adv,
    "mean_backend_percent_advantage": mean_pct,
    "sweet_spot_ranks": rank_list,
    "same_depth_positive_count": same_depth_positive,
    "same_depth_available_count": same_depth_total,
    "seven_day_mean_auc": float(m7["lobo_mean_auc"]),
    "thirty_day_mean_auc": float(m30["lobo_mean_auc"]),
    "seven_day_jaccard": float(m7["avg_top10_jaccard"]),
    "thirty_day_jaccard": float(m30["avg_top10_jaccard"]),
    "sign_test_p_all_positive": float(sign_p),
    "wilcoxon_one_sided_p": None if pd.isna(wilcoxon_p) else float(wilcoxon_p),
    "predicted_vs_hardware_pearson_r": None if pd.isna(pearson[0]) else float(pearson[0]),
    "predicted_vs_hardware_spearman_rho": None if pd.isna(spearman[0]) else float(spearman[0]),
}

with open(FINAL_DIR / "final_evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print("\nSaved final conclusion and JSON summary.")


In [ ]:

# 29. Copy the key existing model-selection artifact into FINAL_RESULTS

for name in [
    "temporal_model_comparison.csv",
    "seven_vs_thirty_day.csv",
]:
    src = locate(name)
    if src is not None:
        df = pd.read_csv(src)
        df.to_csv(TABLE_DIR / name, index=False)

# Save a complete normalized primary dataset.
primary.to_csv(FINAL_DIR / "primary_hardware_dataset.csv", index=False)

print("FINAL RESULTS CREATED")
print("Folder:", FINAL_DIR)
print("Files:")
for p in sorted(FINAL_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(FINAL_DIR))



## Final interpretation checklist

Before using the results in a report/paper, verify:

1. `primary_hardware_dataset.csv` contains exactly **33 rows**: 11 × 3 backends.
2. Kingston rows all use the **fresh** job `dafpntd1ierc738nnmpg`.
3. The old Kingston job `dafgfgl1ierc738nbdd0` does not occur in the primary dataset.
4. Each backend has exactly one Sweet-Spot and ten random controls.
5. All circuits have 256 shots.
6. `FINAL_RESEARCH_CONCLUSION.txt` is generated.
7. All figures are present under `final_results/figures/`.
8. The final claim is framed as **evidence of benefit**, not universal proof.
